# Analisis Spasial & Temporal Data Curah Hujan GSMaP (NetCDF)
Notebook ini digunakan untuk membaca, menggabungkan, dan menganalisis dataset NetCDF (`.nc`) GSMaP dari folder `data/gsmap/` dilengkapi overlay batas kecamatan Kebumen (`33.05_kecamatan.geojson`) serta visualisasi spasial bulanan 2026 dengan `Discrete Boundary Normalization` di folder `gsmap_bulanan/`.

In [ ]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = "DejaVu Sans"
plt.rcParams['font.family'] = "sans-serif"

## 1. Memuat & Menggabungkan File NetCDF GSMaP
Menggunakan `xarray.open_mfdataset` untuk memuat semua file `.nc` dari subfolder `data/gsmap/` secara efisien.

In [ ]:
data_dir = Path("data/gsmap")
nc_files = sorted(list(data_dir.glob("**/*.nc")))
print(f"Total file NetCDF GSMaP ditemukan: {len(nc_files)}")

# Buka seluruh dataset menggunakan xarray
ds = xr.open_mfdataset(nc_files, combine='by_coords')
print("\n--- Informasi Dataset GSMaP ---")
print(ds)

## 2. Memuat Peta Vektor Wilayah (GeoJSON Kecamatan Kebumen)
Membaca file GeoJSON `33.05_kecamatan.geojson` untuk batas administratif dan label nama kecamatan.

In [ ]:
geojson_path = "33.05_kecamatan.geojson"
gdf_kec = gpd.read_file(geojson_path)
print(f"Berhasil memuat GeoJSON: {len(gdf_kec)} Kecamatan di Kebumen")
gdf_kec.head(3)

## 3. Ekstraksi Deret Waktu Rata-rata Wilayah (Areal Mean Time Series)
Merata-ratakan nilai curah hujan di seluruh grid spasial (dimensi `x` dan `y`).

In [ ]:
# Hitung rata-rata spasial (Areal Mean)
da_areal_mean = ds['precipitation'].mean(dim=['x', 'y']).to_series()
df_base = pd.DataFrame({'rainfall': da_areal_mean})
df_base.index.name = 'DateTime'

print("Statistik Data Basis:")
print(df_base.describe())

## 4. Agregasi Waktu (Standar WMO - SUM)
Akumulasi volume curah hujan ke skala Harian (Daily), Bulanan (Monthly), dan Tahunan (Annual).

In [ ]:
if True:
    df_daily = df_base.resample('D').sum()
else:
    df_daily = df_base

df_monthly = df_daily.resample('ME').sum()
df_annual = df_daily.resample('YE').sum()

print(f"Total hari: {len(df_daily)}")
print(f"Total bulan: {len(df_monthly)}")
print(f"Total tahun: {len(df_annual)}")

## 5. Visualisasi Time Series Curah Hujan Harian & Bulanan

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=False)

# Daily plot
axes[0].plot(df_daily.index, df_daily['rainfall'], color='navy', linewidth=0.8, alpha=0.8)
axes[0].set_title("Curah Hujan Harian Rata-rata Wilayah GSMaP", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Curah Hujan (mm/hari)", fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Monthly plot
axes[1].bar(df_monthly.index, df_monthly['rainfall'], color='royalblue', width=20, alpha=0.85)
axes[1].set_title("Akumulasi Curah Hujan Bulanan GSMaP", fontsize=14, fontweight='bold')
axes[1].set_ylabel("Curah Hujan (mm/bulan)", fontsize=12)
axes[1].set_xlabel("Tahun", fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 6. Visualisasi Distribusi Spasial Keseluruhan dengan Overlay Peta Kecamatan GeoJSON
Memetakan sebaran curah hujan raster dan menimpa (overlay) garis batas kecamatan dari `33.05_kecamatan.geojson`.

In [ ]:
# Hitung total akumulasi & rata-rata curah hujan secara spasial
spatial_total = ds['precipitation'].sum(dim='time')
spatial_mean = ds['precipitation'].mean(dim='time')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Peta Total + Overlay GeoJSON
spatial_total.plot(ax=axes[0], cmap='YlGnBu', cbar_kwargs={'label': 'Total Curah Hujan (mm)'})
gdf_kec.boundary.plot(ax=axes[0], color='red', linewidth=1.2, label='Batas Kecamatan')
axes[0].set_title("Peta Total Akumulasi Curah Hujan (GSMaP) - Kebumen", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Bujur (Longitude)", fontsize=11)
axes[0].set_ylabel("Lintang (Latitude)", fontsize=11)
axes[0].legend(loc='upper right')

# Peta Rata-rata + Overlay GeoJSON
spatial_mean.plot(ax=axes[1], cmap='Blues', cbar_kwargs={'label': 'Rata-rata Intensitas (mm/jam)'})
gdf_kec.boundary.plot(ax=axes[1], color='red', linewidth=1.2, label='Batas Kecamatan')
axes[1].set_title("Peta Rata-rata Intensitas Curah Hujan (GSMaP) - Kebumen", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Bujur (Longitude)", fontsize=11)
axes[1].set_ylabel("Lintang (Latitude)", fontsize=11)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 7. Pola Klimatologi Bulanan
Melihat profil rata-rata curah hujan untuk tiap bulan (Januari - Desember) selama periode pengamatan.

In [ ]:
monthly_climatology = df_monthly.groupby(df_monthly.index.month).mean()

plt.figure(figsize=(10, 5))
bars = plt.bar(monthly_climatology.index, monthly_climatology['rainfall'], color='teal', alpha=0.8, edgecolor='black')
plt.title("Klimatologi Curah Hujan Bulanan GSMaP", fontsize=14, fontweight='bold')
plt.xlabel("Bulan", fontsize=12)
plt.ylabel("Curah Hujan (mm/bulan)", fontsize=12)
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des'])
plt.grid(True, axis='y', linestyle='--', alpha=0.7)

# Nilai di atas bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 5, f'{yval:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 8. Visualisasi Spasial Bulanan 2026 dengan Discrete Boundary Normalization
Menghasilkan peta distribusi spasial curah hujan per bulan tahun 2026 berbasis interval diskret ilmiah (`BoundaryNorm`), label nama kecamatan Kebumen, dan colorbar horizontal di bawah.

In [ ]:
# Fungsi Colormap Diskret Ilmiah
def buat_cmap_diskret_ilmiah(bounds_list, nama_palette='YlGnBu'):
    base_cmap = plt.get_cmap(nama_palette)
    norm = mcolors.BoundaryNorm(boundaries=bounds_list, ncolors=base_cmap.N)
    return base_cmap, norm

# Interval curah hujan diskret
interval_hujan = [0, 50, 100, 150, 200, 300, 400, 500, 600, 700, 800]
cmap_ilmiah, norm_ilmiah = buat_cmap_diskret_ilmiah(interval_hujan, 'YlGnBu')

out_dir = Path("{out_folder}")
out_dir.mkdir(exist_ok=True)

# Extent batas Kebumen
minx, miny, maxx, maxy = gdf_kec.total_bounds
pad_x = (maxx - minx) * 0.04
pad_y = (maxy - miny) * 0.04

# Filter dataset khusus 2026
ds_2026 = ds.sel(time='2026')
ds_2026_monthly = ds_2026['precipitation'].groupby('time.month').sum()
months = ds_2026_monthly.month.values

for m in months:
    data_m = ds_2026_monthly.sel(month=m)
    
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Plot raster dengan Discrete Boundary Normalization
    im = data_m.plot(ax=ax, cmap=cmap_ilmiah, norm=norm_ilmiah, add_colorbar=False)
    
    # Overlay batas kecamatan Kebumen
    gdf_kec.boundary.plot(ax=ax, color='black', linewidth=0.7)
    
    # Tuliskan label nama kecamatan
    for _, row in gdf_kec.iterrows():
        pt = row['geometry'].representative_point()
        ax.annotate(
            row['nm_kecamatan'], 
            xy=(pt.x, pt.y),
            fontsize=7.5, 
            fontweight='bold', 
            ha='center', 
            va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='black', linewidth=0.4, alpha=0.85)
        )
        
    # Zoom ke wilayah Kebumen
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    
    # Judul dan Label Sumbu rapi
    ax.set_title(
        f"Peta Akumulasi Curah Hujan Bulanan GSMaP - Kebumen\nBulan: {m:02d} Tahun: 2026", 
        fontsize=13, 
        fontweight='bold', 
        pad=12
    )
    ax.set_xlabel("Longitude", fontsize=11)
    ax.set_ylabel("Latitude", fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Colorbar Tunggal Horizontal di Bawah
    cbar = fig.colorbar(
        im, 
        ax=ax, 
        orientation='horizontal', 
        pad=0.05, 
        shrink=0.85, 
        ticks=interval_hujan
    )
    cbar.set_label('Total Curah Hujan Bulanan (mm)', fontsize=11)
    
    prefix = "gsmap" if "gsmap" in out_folder.lower() else "chirps"
    png_path = out_dir / f"{prefix}_2026_{m:02d}.png"
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.show()
    
print(f"Seluruh peta diskret bulanan 2026 tersimpan di folder: {{out_dir}}")